# 11 · What exactly do our policies do wrong? (behaviour diagnostics, no training)

Success rates and the bowl's final position say *that* our policies fail, not *how*. This notebook re-evaluates three existing checkpoints with per-step recording of the gripper position, the gripper command and the bowl (`p2p/policy/diagnose.py`) and reports, per episode:
* where the gripper was relative to the bowl when it first closed (distance from the bowl's axis; height relative to the rim),
* how often it closed (re-grasp attempts), whether the bowl was lifted, whether it was released,
* whether the policy ended frozen (gripper still for >= 50 steps: a fixed point where the image stops changing).

The same features for the scripted teacher in the same scenes are in `results/act/teacher_behaviour_consistent5.json`.

Checkpoints: `official` (teleop, 97%), `ours_consistent_clean` (20k, 67%), `ours_all` (57%). **Runtime:** T4, ~35 min.

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"],
                   capture_output=True, text=True)
if r.returncode != 0:                                  # show pip's own reason, not just "exit status 1"
    print((r.stdout + r.stderr)[-4000:]); raise RuntimeError("pip install failed: see the lines above")
# Colab preinstalls an older torchao; a newer diffusers (pulled in by lerobot) then fails on import
# (FqnToConfig). ACT does not use torchao, and diffusers skips it when it is absent.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
HUB_REPO = "phone2panda-data"
RUNS = ["act_official_s0", "act_ours_consistent_clean_s0", "act_ours_all_s0"]
OUT = f"{BASE}/behaviour"
from p2p.hub import pull, push
for run in RUNS:
    pull(HUB_REPO, f"{BASE}/hub", allow_patterns=[f"runs/{run}/final/*"])

In [ ]:
for run in RUNS:
    !python -m p2p.policy.act eval {BASE}/hub/runs/{run}/final --episodes 50 --videos 6 --out {OUT}/{run}

In [ ]:
import pandas as pd
for run in RUNS:
    s = json.load(open(f"{OUT}/{run}/eval.json"))
    print(run, s["rate"]); print(json.dumps(s["behaviour"], indent=1))

In [ ]:
push(OUT, HUB_REPO, path_in_repo="behaviour")
!cd {BASE} && zip -qr behaviour.zip behaviour
try:
    from google.colab import files; files.download(f"{BASE}/behaviour.zip")
except Exception:
    print("download", f"{BASE}/behaviour.zip", "from the Output panel")